In [8]:
import yfinance as yf
import pandas as pd
import duckdb

In [95]:
period = '2026-01-01'
spdr_industries = [
    'XLC', 'XLY', 'XLP', 'XLE', 'XLF', 'XLV', 'XLI', 'XLB', 'XLRE', 'XAR',
    'KBE', 'XBI', 'KCE', 'XHE', 'XHS', 'XHB', 'KIE', 'XME', 'XES', 'XOP', 
    'XPH', 'KRE', 'XRT', 'XSD', 'XSW', 'XTL', 'XTN', 'XLK', 'XLU'
]

df_price = yf.download(spdr_industries, start=period)['Close']
df_price.dropna(inplace=True)
df_price = df_price.reset_index()
df_price_long = df_price.melt(
    id_vars=['Date'], 
    var_name='symbol', 
    value_name='price'
)
df_price_long = df_price_long.sort_values(['Date', 'symbol']).reset_index(drop=True)
print(df_price_long.head(10))

[*********************100%***********************]  29 of 29 completed

        Date symbol       price
0 2026-01-02    KBE   60.814716
1 2026-01-02    KCE  150.809708
2 2026-01-02    KIE   59.196697
3 2026-01-02    KRE   64.863297
4 2026-01-02    XAR  250.291672
5 2026-01-02    XBI  121.519997
6 2026-01-02    XES   84.756325
7 2026-01-02    XHB  104.234421
8 2026-01-02    XHE   87.510002
9 2026-01-02    XHS  107.275261


In [97]:
metadata_list = []
for symbol in spdr_industries:
    try:
        info = yf.Ticker(symbol).info
        metadata_list.append({
            'symbol': symbol,
            'name': info.get('longName') or info.get('shortName') or 'N/A',
            'aum': info.get('totalAssets', 0),
            'div_yield': info.get('yield', 0) * 100,
        })
    except Exception as e:
        print(f"Error fetching {symbol}: {e}")

df_metadata = pd.DataFrame(metadata_list)
print(df_metadata.sort_values('aum', ascending=False).head())

   symbol                                             name           aum  \
27    XLK   State Street Technology Select Sector SPDR ETF  124515713024   
4     XLF    State Street Financial Select Sector SPDR ETF   49419214848   
3     XLE       State Street Energy Select Sector SPDR ETF   38671941632   
5     XLV  State Street Health Care Select Sector SPDR ETF   38248288256   
6     XLI   State Street Industrial Select Sector SPDR ETF   30217220096   

    div_yield  
27       0.40  
4        1.54  
3        2.65  
5        1.68  
6        1.18  


In [127]:
ranks = 2
sector_weight = []
for symbol in spdr_industries:
    try:
        ticker = yf.Ticker(symbol)
        weightings = ticker.funds_data.sector_weightings
        if weightings and isinstance(weightings, dict):
            sorted_items = sorted(weightings.items(), 
                                 key=lambda x: x[1], 
                                 reverse=True)[:ranks]
            row = {'symbol': symbol}
            for i, (sector, weight) in enumerate(sorted_items, start=1):
                row[f'top{i}_sector'] = sector
                row[f'top{i}_weight'] = weight * 100
            for i in range(len(sorted_items) + 1, ranks + 1):
                row[f'top{i}_sector'] = None
                row[f'top{i}_weight'] = None
        else:
            row = {'symbol': symbol}
            for i in range(1, ranks + 1):
                row[f'top{i}_sector'] = None
                row[f'top{i}_weight'] = None
        sector_weight.append(row)
    except Exception as e:
        print(f"Error fetching {symbol}: {e}")
        row = {'symbol': symbol}
        for i in range(1, ranks + 1):
            row[f'top{i}_sector'] = None
            row[f'top{i}_weight'] = None
        sector_weight.append(row)

df_top_sector = pd.DataFrame(sector_weight)
print(df_top_sector.head())

  symbol             top1_sector  top1_weight        top2_sector  top2_weight
0    XLC  communication_services       100.00         realestate         0.00
1    XLY       consumer_cyclical        99.01         technology         0.86
2    XLP      consumer_defensive        99.04  consumer_cyclical         0.96
3    XLE                  energy       100.00         realestate         0.00
4    XLF      financial_services        97.99         technology         1.76


In [159]:
df_metadata_extended = duckdb.sql(f"""
SELECT
    a.symbol, name, 
    ROUND(TRY_CAST(aum AS BIGINT) / 1e9, 2) AS aum_bn,
    ROUND(TRY_CAST(div_yield AS DOUBLE), 2) AS div_yield_pct,
    top1_sector AS top1,
    ROUND(TRY_CAST(top1_weight AS DOUBLE), 2) AS top1_pct,
    top2_sector AS top2,
    ROUND(TRY_CAST(top2_weight AS DOUBLE), 2) AS top2_pct,
    ROUND(TRY_CAST(top1_weight AS DOUBLE) + TRY_CAST(top2_weight AS DOUBLE), 2) AS top_pct,
    ROUND(
        100 - (
            TRY_CAST(top1_weight AS DOUBLE) + 
            TRY_CAST(top2_weight AS DOUBLE)
        ),
    2) AS others_pct
FROM df_metadata a
LEFT JOIN df_top_sector b ON a.symbol = b.symbol
ORDER BY aum_bn DESC
LIMIT 5
""")
df_metadata_extended.fetchdf()

,symbol,name,aum_bn,div_yield_pct,top1,top1_pct,top2,top2_pct,top_pct,others_pct
0,XLK,State Street Technology Select Sector SPDR ETF,124.52,0.40,technology,98.97,communication_services,1.03,100.00,0.00
1,XLF,State Street Financial Select Sector SPDR ETF,49.42,1.54,financial_services,97.99,technology,1.76,99.75,0.25
2,XLE,State Street Energy Select Sector SPDR ETF,38.67,2.65,energy,100.00,realestate,0.00,100.00,0.00
3,XLV,State Street Health Care Select Sector SPDR ETF,38.25,1.68,healthcare,100.00,realestate,0.00,100.00,0.00
4,XLI,State Street Industrial Select Sector SPDR ETF,30.22,1.18,industrials,93.84,technology,5.91,99.75,0.25
